In [8]:
import numpy as np
from dataclasses import dataclass
from sklearn.decomposition import KernelPCA
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# -----------------------
# Lorenz-96 simulator
# -----------------------
def lorenz96_rhs(x, F=8.0):
    return (np.roll(x, -1) - np.roll(x, 2)) * np.roll(x, 1) - x + F

def rk4_step(x, dt, F=8.0):
    k1 = lorenz96_rhs(x, F)
    k2 = lorenz96_rhs(x + 0.5 * dt * k1, F)
    k3 = lorenz96_rhs(x + 0.5 * dt * k2, F)
    k4 = lorenz96_rhs(x + dt * k3, F)
    return x + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)

def simulate_l96(N=40, F=8.0, dt=0.01, steps=6000, burn_in=1000, seed=0):
    rng = np.random.default_rng(seed)
    x = F * np.ones(N) + 0.01 * rng.standard_normal(N)
    for _ in range(burn_in):
        x = rk4_step(x, dt, F)
    T = steps - burn_in
    X = np.empty((T, N), dtype=np.float64)
    for t in range(T):
        x = rk4_step(x, dt, F)
        X[t] = x
    return X

# -----------------------
# Helpers: subsampling
# -----------------------
def subsample_rows(A, max_rows, rng):
    n = A.shape[0]
    if max_rows is None or max_rows <= 0 or n <= max_rows:
        return A
    idx = rng.choice(n, size=max_rows, replace=False)
    return A[idx]

def make_time_subsample_indices(T, stride, rng, max_rows=None):
    idx = np.arange(0, T, stride, dtype=int)
    if max_rows is not None and idx.size > max_rows:
        idx = rng.choice(idx, size=max_rows, replace=False)
        idx.sort()
    return idx

def median_heuristic_gamma(X, rng, max_pairs=20000):
    """
    Returns gamma = 1 / (2 * median(||xi-xj||^2)) computed from random pairs.
    Works in any dimension. Assumes X is already scaled appropriately.
    """
    n = X.shape[0]
    if n < 2:
        return 1.0
    # sample pairs
    m = min(max_pairs, n * (n - 1) // 2)
    i = rng.integers(0, n, size=m)
    j = rng.integers(0, n, size=m)
    mask = i != j
    i, j = i[mask], j[mask]
    if i.size == 0:
        return 1.0
    d2 = np.sum((X[i] - X[j]) ** 2, axis=1)
    med = np.median(d2[d2 > 0]) if np.any(d2 > 0) else np.median(d2)
    if not np.isfinite(med) or med <= 0:
        return 1.0
    return 1.0 / (2.0 * med)

# -----------------------
# Foundation SSM config
# -----------------------
@dataclass
class Config:
    N: int = 40
    F: float = 8.0
    dt: float = 0.01
    steps: int = 6000
    burn_in: int = 1000

    d_latent: int = 12
    n_train_traj: int = 20
    n_test_traj: int = 10
    train_frac: float = 0.8

    rollout_horizon: int = 200

    # sampling controls
    train_stride: int = 10
    max_kpca_snapshots: int = 4000
    max_dyn_pairs: int = 5000
    max_dec_pairs: int = 5000

    # regularization
    F_alpha: float = 1e-2
    g_alpha: float = 1e-3  # ridge for linear decoder

    # rollout stabilization (0 disables)
    rollout_relax: float = 0.15   # try 0.0, 0.1, 0.2

# -----------------------
# Training
# -----------------------
def fit_foundation_ssm(cfg: Config, seed=0):
    rng = np.random.default_rng(seed)

    # 1) Generate training trajectories
    trajs = [simulate_l96(cfg.N, cfg.F, cfg.dt, cfg.steps, cfg.burn_in, seed=seed + m)
             for m in range(cfg.n_train_traj)]

    # 2) Pool snapshots for kPCA (subsampled)
    X_pool_list = []
    for X in trajs:
        split = int(cfg.train_frac * len(X))
        X_tr = X[:split]
        idx = make_time_subsample_indices(len(X_tr), cfg.train_stride, rng)
        X_pool_list.append(X_tr[idx])
    X_pool = np.vstack(X_pool_list)
    X_pool = subsample_rows(X_pool, cfg.max_kpca_snapshots, rng)

    # Scale x before kernels
    x_scaler = StandardScaler(with_mean=True, with_std=True)
    X_pool_scaled = x_scaler.fit_transform(X_pool)

    # --- EDIT 1: choose kpca_gamma from data ---
    kpca_gamma = median_heuristic_gamma(X_pool_scaled, rng)

    encoder = KernelPCA(
        n_components=cfg.d_latent,
        kernel="rbf",
        gamma=kpca_gamma,
        fit_inverse_transform=False,
        random_state=seed,
    )
    encoder.fit(X_pool_scaled)

    # 3) Build datasets
    S_in_list, S_out_list = [], []
    S_dec_list, X_dec_list = [], []

    for X in trajs:
        split = int(cfg.train_frac * len(X))
        X_tr = X[:split]

        idx = make_time_subsample_indices(len(X_tr), cfg.train_stride, rng)
        X_sub = X_tr[idx]

        X_sub_scaled = x_scaler.transform(X_sub)
        S_sub = encoder.transform(X_sub_scaled)

        if len(S_sub) >= 2:
            S_in_list.append(S_sub[:-1])
            S_out_list.append(S_sub[1:])
        S_dec_list.append(S_sub)
        X_dec_list.append(X_sub)

    S_in = np.vstack(S_in_list)
    S_out = np.vstack(S_out_list)
    S_dec = np.vstack(S_dec_list)
    X_dec = np.vstack(X_dec_list)

    # cap rows
    dyn = np.hstack([S_in, S_out])
    dyn = subsample_rows(dyn, cfg.max_dyn_pairs, rng)
    S_in = dyn[:, :cfg.d_latent]
    S_out = dyn[:, cfg.d_latent:]

    dec = np.hstack([S_dec, X_dec])
    dec = subsample_rows(dec, cfg.max_dec_pairs, rng)
    S_dec = dec[:, :cfg.d_latent]
    X_dec = dec[:, cfg.d_latent:]


    # --- EDIT 3: choose F_gamma from latent data ---
    F_gamma = 0.1
    F_model = KernelRidge(kernel="rbf", gamma=F_gamma, alpha=cfg.F_alpha).fit(S_in, S_out)

    F_model.fit(S_in, S_out)


    # Keep linear decoder (usually more stable/cheap)
    g_model = Ridge(alpha=1e-3).fit(S_dec, X_dec)

    # return gammas too so you can print/debug
    
    info = {"kpca_gamma": kpca_gamma, "F_gamma": F_gamma}
    return x_scaler, s_scaler, encoder, F_model, g_model, info

# -----------------------
# Forecasting / evaluation
# -----------------------
def rollout(cfg: Config, x_scaler, s_scaler, encoder, F_model, g_model, x0, H):
    x0_scaled = x_scaler.transform(x0.reshape(1, -1))
    s = encoder.transform(x0_scaled)       # (1, d) in raw latent coords
    s = s_scaler.transform(s)              # (1, d) standardized latent

    preds = np.empty((H, x0.size), dtype=np.float64)
    for t in range(H):
        preds[t] = g_model.predict(s)[0]
        s_next = F_model.predict(s)

        # --- EDIT 4: optional relaxation to reduce off-manifold drift ---
        if cfg.rollout_relax and cfg.rollout_relax > 0:
            s = (1.0 - cfg.rollout_relax) * s + cfg.rollout_relax * s_next
        else:
            s = s_next

    return preds

def evaluate_on_unseen(cfg: Config, x_scaler, s_scaler, encoder, F_model, g_model, seed=10):
    one_step_mses, rollout_mses = [], []

    for j in range(cfg.n_test_traj):
        X = simulate_l96(cfg.N, cfg.F, cfg.dt, cfg.steps, cfg.burn_in, seed=seed + j)
        split = int(cfg.train_frac * len(X))
        X_te_full = X[split:]
        idx = np.arange(0, len(X_te_full), cfg.train_stride)
        X_te = X_te_full[idx]
        # one-step
        X_te_scaled = x_scaler.transform(X_te)
        S_te = encoder.transform(X_te_scaled)
        S_te_s = s_scaler.transform(S_te)

        S_pred_s = F_model.predict(S_te_s[:-1])
        X_pred_next = g_model.predict(S_pred_s)
        one_step_mses.append(mean_squared_error(X_te[1:], X_pred_next))

        # rollout
        H = cfg.rollout_horizon
        if len(X_te) <= H + 1:
            raise ValueError("Test segment too short for chosen rollout horizon.")
        X_hat = rollout(cfg, x_scaler, s_scaler, encoder, F_model, g_model, X_te[0], H)
        rollout_mses.append(mean_squared_error(X_te[:H], X_hat))

    return (float(np.mean(one_step_mses)), float(np.std(one_step_mses)),
            float(np.mean(rollout_mses)), float(np.std(rollout_mses)))

if __name__ == "__main__":
    cfg = Config(
        d_latent=12,
        n_train_traj=20,
        n_test_traj=10,
        steps=6000,
        burn_in=1000,
        rollout_horizon=20,
        train_stride=10,
        max_kpca_snapshots=4000,
        max_dyn_pairs=5000,
        max_dec_pairs=5000,
        F_alpha=1e-2,
        g_alpha=1e-3,
        rollout_relax=0.0,
    )

    x_scaler, s_scaler, encoder, F_model, g_model, info = fit_foundation_ssm(cfg, seed=0)
    print("Chosen gammas:", info)

    mse1_mean, mse1_std, mseR_mean, mseR_std = evaluate_on_unseen(cfg, x_scaler, s_scaler, encoder, F_model, g_model)
    print(f"Unseen trajectories: 1-step x-space MSE = {mse1_mean:.4e} ± {mse1_std:.4e}")
    print(f"Unseen trajectories: {cfg.rollout_horizon}-step rollout x-space MSE = {mseR_mean:.4e} ± {mseR_std:.4e}")


Chosen gammas: {'kpca_gamma': np.float64(0.006263004533932027), 'F_gamma': 0.1}
Unseen trajectories: 1-step x-space MSE = 3.5547e+01 ± 1.4819e+00
Unseen trajectories: 20-step rollout x-space MSE = 3.9508e+01 ± 5.7921e+00
